# NLP Lab 4: Recurrent Neural Networks (RNNs, LSTMs) with PyTorch

Welcome to Lab 4! In this lab, we will use **PyTorch** and **Gensim** to build recurrent architectures across fundamental sequence tasks.

### **Lab Objectives:**
1. **Pretrained Word Embeddings:** Train a Skip-gram Word2Vec model using Gensim (`sg=1`) and load weights into PyTorch (`nn.Embedding.from_pretrained`).
2. **Sequence Classification:** Compare a **Vanilla RNN** vs. a **Stacked Bidirectional LSTM**.
3. **Sequence Labeling:** Build a **Bidirectional RNN** for Part-of-Speech (POS) tagging.
4. **Language Modeling:** Train a **Stacked Multi-Layer LSTM** on Shakespeare text to generate new sentences.
5. **Encoder-Decoder Seq2Seq:** Build an **English-to-Bengali Machine Translation** model.

In [7]:
%pip install torch torchvision torchaudio
%pip install gensim
%pip install urllib3


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import gensim.downloader as api
import numpy as np
import random
import urllib.request
from gensim.models import Word2Vec

# Set reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


---
## Topic 0: Skip-Gram Word2Vec Embedding Integration

Instead of building Skip-gram from scratch or using random embeddings, we use Google's official pre-trained Word2Vec model (word2vec-google-news-300) via Gensim's downloader API instead of training a model from scratch. We then extract the weight matrix and load it directly into PyTorch's `nn.Embedding.from_pretrained()` layer.

In [ ]:
def build_pretrained_embedding_matrix(corpus, w2v_model=None, model_name='word2vec-google-news-300'):
    """
    Builds a PyTorch vocabulary and embedding matrix from a target corpus using Google's pre-trained Word2Vec.
    
    Args:
        corpus (list of list of str): Tokenized corpus (e.g., [["i", "love", "nlp"], ...]).
        w2v_model (KeyedVectors, optional): Pre-loaded Gensim model. If None, loads from `model_name`.
        model_name (str): Gensim model identifier if `w2v_model` is not passed.
        
    Returns:
        vocab (dict): Word-to-index mapping with <PAD>: 0 and <UNK>: 1.
        embedding_matrix (torch.Tensor): Tensor of shape (vocab_size, embedding_dim).
    """
    # Load model if not already provided
    if w2v_model is None:
        print(f"Loading '{model_name}'... (This may take a moment)")
        w2v_model = api.load(model_name)
        
    embedding_dim = w2v_model.vector_size
    
    # Initialize vocabulary with PAD and UNK tokens
    vocab = {"<PAD>": 0, "<UNK>": 1}
    
    # Index 0 -> Zeros for PAD; Index 1 -> Random normal vector for UNK
    embedding_weights = [np.zeros(embedding_dim), np.random.randn(embedding_dim)]
    
    # Extract embeddings for words in corpus
    for sentence in corpus:
        for word in sentence:
            if word not in vocab:
                vocab[word] = len(vocab)
                
                if word in w2v_model:
                    embedding_weights.append(w2v_model[word])
                else:
                    # Random vector for out-of-vocabulary (OOV) tokens
                    embedding_weights.append(np.random.randn(embedding_dim))
                    
    embedding_matrix = torch.tensor(np.array(embedding_weights), dtype=torch.float32)
    
    return vocab, embedding_matrix

In [ ]:
# 1. Load Google's model once (prevents reloading every time)
print("Loading Google Word2Vec Model...")
w2v_google = api.load('word2vec-google-news-300')

# 2. Define target corpus
corpus = [
    ["i", "love", "this", "fantastic", "movie"],
    ["great", "film", "amazing", "acting"],
    ["wonderful", "experience", "loved", "it"],
    ["terrible", "waste", "of", "time"],
    ["horrible", "script", "bad", "director"],
    ["boring", "ugly", "awful", "movie"]
]

# 3. Call the function
vocab, embedding_matrix = build_pretrained_embedding_matrix(corpus, w2v_model=w2v_google)

print(f"Vocabulary Size: {len(vocab)}")
print(f"Embedding Matrix Shape: {embedding_matrix.shape}")

Loading Google Word2Vec Model...
[==================================================] 100.0% 1662.8/1662.8MB downloaded
Vocabulary Size: 26
Embedding Matrix Shape: torch.Size([26, 300])
Embedding Layer Output Dim: 300


---
## Topic 1: Sequence Classification (Vanilla RNN vs. Stacked BiLSTM)

Here we explicitly contrast **Vanilla `nn.RNN`** with a **Stacked Bidirectional `nn.LSTM`**.
* **Vanilla RNN:** Computes $h_t = \tanh(W_{ih} x_t + b_{ih} + W_{hh} h_{t-1} + b_{hh})$. It returns only $h_n$.
* **LSTM:** Adds a cell memory state $c_t$ to mitigate vanishing gradients. It returns $(h_n, c_n)$.

In [13]:
from sklearn.model_selection import train_test_split

# 1. Expanded Sentiment Dataset
sentences = [
    # Positive Sentences (Label: 1)
    "i love this fantastic movie", 
    "great film amazing acting", 
    "wonderful experience loved it",
    "brilliant cinema enjoyed every minute", 
    "absolute masterpiece highly recommend",
    "excellent performance by the cast", 
    "superb direction and great story",
    "loved the movie it was awesome", 
    "fantastic plot and brilliant soundtrack",
    "a delightful and heartwarming film",
    
    # Negative Sentences (Label: 0)
    "terrible waste of time", 
    "horrible script bad director", 
    "boring ugly awful movie",
    "hated it worst film ever", 
    "dull acting and poor plot", 
    "total disaster waste of money",
    "disappointing experience very boring", 
    "bad performance predictable story",
    "awful direction terribly slow", 
    "lacks depth completely uninspiring"
]

labels = [1] * 10 + [0] * 10  # 10 Positive (1), 10 Negative (0)

# 2. Extract Corpus & Update Vocabulary / Embedding Matrix
corpus = [s.split() for s in sentences]

# Using our custom function to build embeddings from Google's Word2Vec
vocab, embedding_matrix = build_pretrained_embedding_matrix(corpus, w2v_model=w2v_google)

# 3. Encoding & Padding Function
def encode(sent, vocab, max_len=7):
    tokens = [vocab.get(w, vocab["<UNK>"]) for w in sent.split()]
    return tokens[:max_len] + [vocab["<PAD>"]] * max(0, max_len - len(tokens))

# Encode all dataset samples
X_all = torch.tensor([encode(s, vocab) for s in sentences], dtype=torch.long)
y_all = torch.tensor(labels, dtype=torch.float32)

# 4. Train / Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=SEED, stratify=y_all
)

# Move tensors to hardware device
X_train, y_train = X_train.to(device), y_train.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)

print(f"Total Samples: {len(sentences)}")
print(f"Train Set Shape: {X_train.shape} | Test Set Shape: {X_test.shape}")

# 5. Model Architectures
class VanillaRNNClassifier(nn.Module):
    def __init__(self, embed_matrix, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(embed_matrix, freeze=False, padding_idx=0)
        self.rnn = nn.RNN(embed_matrix.shape[1], hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        
    def forward(self, x):
        embedded = self.embedding(x)
        out, h_n = self.rnn(embedded)
        return self.fc(h_n.squeeze(0)).squeeze(1)

class StackedBiLSTMClassifier(nn.Module):
    def __init__(self, embed_matrix, hidden_dim, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(embed_matrix, freeze=False, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_matrix.shape[1], hidden_dim, 
            num_layers=num_layers, batch_first=True, bidirectional=True
        )
        self.fc = nn.Linear(hidden_dim * 2, 1)
        
    def forward(self, x):
        embedded = self.embedding(x)
        out, (h_n, c_n) = self.lstm(embedded)
        forward_h = h_n[-2, :, :]
        backward_h = h_n[-1, :, :]
        bi_hidden = torch.cat((forward_h, backward_h), dim=1)
        return self.fc(bi_hidden).squeeze(1)

# 6. Evaluation Function for Test Set
def evaluate(model, X_eval, y_eval, criterion):
    model.eval()
    with torch.no_grad():
        logits = model(X_eval)
        loss = criterion(logits, y_eval).item()
        probs = torch.sigmoid(logits)
        predictions = (probs >= 0.5).float()
        accuracy = (predictions == y_eval).float().mean().item() * 100
    return loss, accuracy

# 7. Model Instantiation & Training Loss
criterion = nn.BCEWithLogitsLoss()

# --- Train Model A: Vanilla RNN ---
rnn_model = VanillaRNNClassifier(embedding_matrix, hidden_dim=16).to(device)
opt_rnn = optim.Adam(rnn_model.parameters(), lr=0.01)

print(rnn_model)

for epoch in range(80):
    rnn_model.train()
    opt_rnn.zero_grad()
    y_pred_rnn = rnn_model(X_train)
    loss_rnn = criterion(y_pred_rnn, y_train)
    loss_rnn.backward()
    opt_rnn.step()

# --- Train Model B: Stacked BiLSTM ---
bilstm_model = StackedBiLSTMClassifier(embedding_matrix, hidden_dim=16, num_layers=2).to(device)
opt_lstm = optim.Adam(bilstm_model.parameters(), lr=0.01)

print(bilstm_model)

for epoch in range(80):
    bilstm_model.train()
    opt_lstm.zero_grad()
    y_pred_lstm = bilstm_model(X_train)
    loss_lstm = criterion(y_pred_lstm, y_train)
    loss_lstm.backward()
    opt_lstm.step()

# 8. Evaluate both models on Test Set
test_loss_rnn, test_acc_rnn = evaluate(rnn_model, X_test, y_test, criterion)
test_loss_lstm, test_acc_lstm = evaluate(bilstm_model, X_test, y_test, criterion)

print("\n================ EVALUATION RESULTS ================")
print(f"Vanilla RNN     | Test Loss: {test_loss_rnn:.4f} | Test Accuracy: {test_acc_rnn:.2f}%")
print(f"Stacked BiLSTM  | Test Loss: {test_loss_lstm:.4f} | Test Accuracy: {test_acc_lstm:.2f}%")
print("====================================================")

Total Samples: 20
Train Set Shape: torch.Size([16, 7]) | Test Set Shape: torch.Size([4, 7])
VanillaRNNClassifier(
  (embedding): Embedding(68, 300, padding_idx=0)
  (rnn): RNN(300, 16, batch_first=True)
  (fc): Linear(in_features=16, out_features=1, bias=True)
)
StackedBiLSTMClassifier(
  (embedding): Embedding(68, 300, padding_idx=0)
  (lstm): LSTM(300, 16, num_layers=2, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=32, out_features=1, bias=True)
)

================ EVALUATION RESULTS ================
Vanilla RNN     | Test Loss: 1.9474 | Test Accuracy: 25.00%
Stacked BiLSTM  | Test Loss: 2.9353 | Test Accuracy: 50.00%


In [16]:
# Function to predict sentiment for any single input sentence
def predict_sentiment(sentence, model, vocab, max_len=7):
    model.eval()  # Set model to evaluation mode
    
    # 1. Preprocess and encode the sentence into tensor format
    encoded = encode(sentence, vocab, max_len=max_len)
    input_tensor = torch.tensor([encoded], dtype=torch.long).to(device)  # Batch shape: (1, max_len)
    
    # 2. Forward pass through model
    with torch.no_grad():
        logits = model(input_tensor)
        prob = torch.sigmoid(logits).item()  # Convert logit to probability [0, 1]
        
    # 3. Determine sentiment class and confidence
    prediction = "Positive" if prob >= 0.5 else "Negative"
    confidence = prob if prob >= 0.5 else (1 - prob)
    
    return prediction, confidence, prob

# ======================================================
# --- TEST YOUR SINGLE SENTENCE HERE ---
# ======================================================
sample_sentence = "the acting was brilliant and absolutely fantastic"
# Try other sentences:
#sample_sentence = "it was a terrible and complete waste of time"

print(f"Input Sentence: '{sample_sentence}'\n" + "-"*55)

# Predict using Vanilla RNN
rnn_pred, rnn_conf, rnn_prob = predict_sentiment(sample_sentence, rnn_model, vocab)
print(f"Vanilla RNN     | Prediction: {rnn_pred:<8} | Confidence: {rnn_conf*100:.2f}% (Prob: {rnn_prob:.4f})")

# Predict using Stacked BiLSTM
bilstm_pred, bilstm_conf, bilstm_prob = predict_sentiment(sample_sentence, bilstm_model, vocab)
print(f"Stacked BiLSTM  | Prediction: {bilstm_pred:<8} | Confidence: {bilstm_conf*100:.2f}% (Prob: {bilstm_prob:.4f})")

Input Sentence: 'the acting was brilliant and absolutely fantastic'
-------------------------------------------------------
Vanilla RNN     | Prediction: Negative | Confidence: 68.46% (Prob: 0.3154)
Stacked BiLSTM  | Prediction: Positive | Confidence: 99.96% (Prob: 0.9996)


---
## Topic 2: Sequence Labeling with Bidirectional RNN

In sequence labeling (such as POS tagging), predictions are made at every time step $t$. Here we use a **Bidirectional Vanilla RNN** (`nn.RNN(..., bidirectional=True)`).

In [9]:
# POS Dataset
raw_data = [
    (["i", "love", "this", "movie"], ["PRON", "VERB", "DET", "NOUN"]),
    (["great", "acting", "and", "script"], ["ADJ", "NOUN", "CONJ", "NOUN"])
]

tag_to_ix = {"<PAD>": 0, "PRON": 1, "VERB": 2, "DET": 3, "NOUN": 4, "ADJ": 5, "CONJ": 6}
ix_to_tag = {v: k for k, v in tag_to_ix.items()}

# Bidirectional Vanilla RNN Sequence Labeler
class BiRNNSequenceLabeler(nn.Module):
    def __init__(self, embed_matrix, hidden_dim, tagset_size):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(embed_matrix, freeze=False, padding_idx=0)
        # Bidirectional Vanilla RNN
        self.rnn = nn.RNN(embed_matrix.shape[1], hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, tagset_size)
        
    def forward(self, x):
        embeds = self.embedding(x)
        rnn_out, _ = self.rnn(embeds) # (batch, seq_len, hidden_dim * 2)
        logits = self.fc(rnn_out)      # (batch, seq_len, tagset_size)
        return logits

# Encode inputs
X_pos = torch.tensor([[vocab.get(w, 1) for w in seq[0]] for seq in raw_data], dtype=torch.long).to(device)
y_pos = torch.tensor([[tag_to_ix[t] for t in seq[1]] for seq in raw_data], dtype=torch.long).to(device)

print(f"POS Input Sequences: {X_pos}")
print(f"POS Labels: {y_pos}")

# Train Labeler
pos_model = BiRNNSequenceLabeler(embedding_matrix, hidden_dim=16, tagset_size=len(tag_to_ix)).to(device)
pos_criterion = nn.CrossEntropyLoss(ignore_index=0)
pos_opt = optim.Adam(pos_model.parameters(), lr=0.02)

print(pos_model)

for epoch in range(100):
    pos_model.train()
    logits = pos_model(X_pos)
    loss = pos_criterion(logits.view(-1, len(tag_to_ix)), y_pos.view(-1))
    pos_opt.zero_grad()
    loss.backward()
    pos_opt.step()

print(f"BiRNN Sequence Labeler Trained! Loss: {loss.item():.4f}")

# Inference
test_sent = ["great", "script"]
test_in = torch.tensor([[vocab.get(w, 1) for w in test_sent]], dtype=torch.long).to(device)
with torch.no_grad():
    preds = pos_model(test_in).argmax(dim=-1).squeeze(0).tolist()
    print("Predicted Tags:", [ix_to_tag[p] for p in preds])

POS Input Sequences: tensor([[ 2,  3,  4,  6],
        [ 7, 10, 31, 45]], device='cuda:0')
POS Labels: tensor([[1, 2, 3, 4],
        [5, 4, 6, 4]], device='cuda:0')
BiRNNSequenceLabeler(
  (embedding): Embedding(68, 300, padding_idx=0)
  (rnn): RNN(300, 16, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=32, out_features=7, bias=True)
)
BiRNN Sequence Labeler Trained! Loss: 0.0002
Predicted Tags: ['ADJ', 'NOUN']


---
## Topic 3: Language Modeling & Sentence Generation with Stacked LSTM

We automatically download the **Tiny Shakespeare** corpus, train a **2-Layer Stacked LSTM Language Model**, and use autoregressive sampling to generate text.

In [10]:
import re
from torch.utils.data import TensorDataset

# 1. Download Corpus & Perform Word Tokenization
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
raw_text = urllib.request.urlopen(url).read().decode('utf-8')[:30000] # Expanded character slice

# Extract word tokens using regular expressions (lowercased)
words = re.findall(r"\b\w+\b", raw_text.lower())

# Group words into chunks to pass into our build_pretrained_embedding_matrix function
chunk_size = 50
word_chunks = [words[i:i + chunk_size] for i in range(0, len(words), chunk_size)]

# 2. Extract Vocabulary & Google Word2Vec Embedding Matrix (300-dim)
vocab, embedding_matrix = build_pretrained_embedding_matrix(word_chunks, w2v_model=w2v_google)
idx2word = {v: k for k, v in vocab.items()}
word_vocab_size = len(vocab)

# 3. Build Word-Level Sequence Data (Sliding Window)
seq_length = 5  # Sliding window of 5 words
lm_X, lm_Y = [], []
encoded_words = [vocab.get(w, vocab["<UNK>"]) for w in words]

for i in range(0, len(encoded_words) - seq_length):
    lm_X.append(encoded_words[i : i + seq_length])
    lm_Y.append(encoded_words[i + 1 : i + seq_length + 1])

lm_X_tensor = torch.tensor(lm_X, dtype=torch.long)
lm_Y_tensor = torch.tensor(lm_Y, dtype=torch.long)

lm_loader = DataLoader(TensorDataset(lm_X_tensor, lm_Y_tensor), batch_size=64, shuffle=True)

# 4. Stacked 2-Layer LSTM Language Model using Pre-trained Embeddings
class StackedLSTMLanguageModel(nn.Module):
    def __init__(self, embed_matrix, hidden_dim=128, num_layers=2):
        super().__init__()
        # Load pre-trained Google Word2Vec weights into Embedding layer
        self.embedding = nn.Embedding.from_pretrained(embed_matrix, freeze=False, padding_idx=0)
        embed_dim = embed_matrix.shape[1] # 300 dimensions
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, embed_matrix.shape[0])
        
    def forward(self, x, hidden=None):
        embeds = self.embedding(x)
        out, hidden = self.lstm(embeds, hidden)
        logits = self.fc(out)
        return logits, hidden

lm_model = StackedLSTMLanguageModel(embedding_matrix, hidden_dim=128, num_layers=2).to(device)
lm_optimizer = optim.Adam(lm_model.parameters(), lr=0.003)
lm_criterion = nn.CrossEntropyLoss(ignore_index=0)

print(lm_model)

# Training loop
for epoch in range(12):
    lm_model.train()
    total_loss = 0
    for bx, by in lm_loader:
        bx, by = bx.to(device), by.to(device)
        logits, _ = lm_model(bx)
        loss = lm_criterion(logits.view(-1, word_vocab_size), by.view(-1))
        lm_optimizer.zero_grad()
        loss.backward()
        lm_optimizer.step()
        total_loss += loss.item()
    
    if (epoch + 1) % 4 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/12 - Loss: {total_loss/len(lm_loader):.4f}")

# 5. Autoregressive Word-Level Sentence Generation Function
def generate_sentences(model, prompt="first citizen", generate_length=20, temperature=0.7):
    model.eval()
    prompt_words = re.findall(r"\b\w+\b", prompt.lower())
    input_indices = [vocab.get(w, vocab["<UNK>"]) for w in prompt_words]
    input_eval = torch.tensor([input_indices], dtype=torch.long).to(device)
    
    generated_words = list(prompt_words)
    hidden = None
    
    with torch.no_grad():
        for _ in range(generate_length):
            logits, hidden = model(input_eval, hidden)
            # Predict the next word based on the final position logits
            last_logits = logits[0, -1, :] / temperature
            probs = torch.softmax(last_logits, dim=-1)
            
            pred_idx = torch.multinomial(probs, num_samples=1).item()
            
            input_eval = torch.tensor([[pred_idx]], dtype=torch.long).to(device)
            generated_words.append(idx2word.get(pred_idx, "<UNK>"))
            
    return " ".join(generated_words)

print("\n--- Generated Shakespeare Text ---")
print(generate_sentences(lm_model, prompt="first citizen", generate_length=25))

StackedLSTMLanguageModel(
  (embedding): Embedding(1412, 300, padding_idx=0)
  (lstm): LSTM(300, 128, num_layers=2, batch_first=True)
  (fc): Linear(in_features=128, out_features=1412, bias=True)
)
Epoch 1/12 - Loss: 6.2910
Epoch 4/12 - Loss: 4.5285
Epoch 8/12 - Loss: 2.0971
Epoch 12/12 - Loss: 1.1023

--- Generated Shakespeare Text ---
first citizen very well and could be none yet valeria verily i do not jest with you my good madam i should tell you excellent news of


---
## Topic 4: Encoder-Decoder Seq2Seq Machine Translation (English to Bengali)

In an **Encoder-Decoder (Seq2Seq)** model:
1. **Encoder (LSTM):** Processes the input English sequence and compresses it into hidden context vectors $(h, c)$.
2. **Decoder (LSTM):** Takes the context vectors as initial states and generates Bengali tokens autoregressively.

In [23]:
# 1. Parallel Dataset (English -> Bengali)
pairs = [
    ("i love my country", "আমি আমার দেশকে ভালোবাসি"),
    ("how are you", "আপনি কেমন আছেন"),
    ("good morning", "সুপ্রভাত"),
    ("what is your name", "আপনার নাম কি"),
    ("thank you very much", "আপনাকে অনেক ধন্যবাদ"),
    ("i am learning nlp", "আমি এনএলপি শিখছি"),
    ("this is a good book", "এটি একটি ভালো বই"),
    ("the cat drinks milk", "বিড়ালটি দুধ খায়"),
    ("we are very happy", "আমরা খুব খুশি"),
    ("see you again", "আবার দেখা হবে")
]

# Build Vocabularies
en_vocab = {"<PAD>": 0, "<UNK>": 1, "<SOS>": 2, "<EOS>": 3}
bn_vocab = {"<PAD>": 0, "<UNK>": 1, "<SOS>": 2, "<EOS>": 3}

for en_sent, bn_sent in pairs:
    for word in en_sent.split():
        if word not in en_vocab: en_vocab[word] = len(en_vocab)
    for word in bn_sent.split():
        if word not in bn_vocab: bn_vocab[word] = len(bn_vocab)

inv_bn_vocab = {v: k for k, v in bn_vocab.items()}

# Helper Encoders
def encode_seq(sent, vocab):
    return [vocab["<SOS>"]] + [vocab.get(w, vocab["<UNK>"]) for w in sent.split()] + [vocab["<EOS>"]]

# Pad Sequences
max_en_len = max([len(s.split()) for s, _ in pairs]) + 2
max_bn_len = max([len(s.split()) for _, s in pairs]) + 2

en_tensors, bn_tensors = [], []
for en_s, bn_s in pairs:
    e_enc = encode_seq(en_s, en_vocab)
    b_enc = encode_seq(bn_s, bn_vocab)
    
    e_enc += [en_vocab["<PAD>"]] * (max_en_len - len(e_enc))
    b_enc += [bn_vocab["<PAD>"]] * (max_bn_len - len(b_enc))
    
    en_tensors.append(e_enc)
    bn_tensors.append(b_enc)

X_seq2seq = torch.tensor(en_tensors, dtype=torch.long).to(device)
Y_seq2seq = torch.tensor(bn_tensors, dtype=torch.long).to(device)

# ==============================================================================
# Build Pre-trained Google Word2Vec Embedding Matrix for English Encoder
# ==============================================================================
en_embed_dim = 300
en_embedding_matrix = torch.zeros((len(en_vocab), en_embed_dim))

for word, idx in en_vocab.items():
    if word in w2v_google:
        en_embedding_matrix[idx] = torch.tensor(w2v_google[word])
    else:
        if word != "<PAD>":
            # Random initialization for special tokens or Out-Of-Vocabulary words
            en_embedding_matrix[idx] = torch.randn(en_embed_dim) * 0.1

en_embedding_matrix = en_embedding_matrix.to(device)


# 2. Seq2Seq Architecture with Pretrained Word2Vec Encoder
class Seq2SeqEncoder(nn.Module):
    def __init__(self, embed_matrix, hidden_dim):
        super().__init__()
        # Load pre-trained Google Word2Vec weights into Embedding layer
        self.embedding = nn.Embedding.from_pretrained(embed_matrix, freeze=False, padding_idx=0)
        embed_dim = embed_matrix.shape[1] # 300 dimensions
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        
    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, cell) = self.lstm(embedded)
        return hidden, cell

class Seq2SeqDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        # Bengali embedding layer (trainable)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        
    def forward(self, x, hidden, cell):
        embedded = self.embedding(x)
        out, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        preds = self.fc(out)
        return preds, hidden, cell

class Seq2SeqTranslation(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        
    def forward(self, src, trg):
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.fc.out_features
        
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(device)
        hidden, cell = self.encoder(src)
        
        # Teacher forcing: feed actual targets iteratively
        decoder_input = trg[:, 0].unsqueeze(1)
        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(decoder_input, hidden, cell)
            outputs[:, t, :] = output.squeeze(1)
            decoder_input = trg[:, t].unsqueeze(1)
            
        return outputs

# Initialize & Train
encoder = Seq2SeqEncoder(en_embedding_matrix, hidden_dim=64)
decoder = Seq2SeqDecoder(len(bn_vocab), embed_dim=32, hidden_dim=64)
translator = Seq2SeqTranslation(encoder, decoder).to(device)

opt_translator = optim.Adam(translator.parameters(), lr=0.01)
criterion_seq2seq = nn.CrossEntropyLoss(ignore_index=0)

print(f"Encoder: {encoder}")
print(f"Decoder: {decoder}")
print(f"Translator: {translator}")

for epoch in range(120):
    translator.train()
    output = translator(X_seq2seq, Y_seq2seq)
    loss = criterion_seq2seq(output[:, 1:, :].reshape(-1, len(bn_vocab)), Y_seq2seq[:, 1:].reshape(-1))
    opt_translator.zero_zero = None
    opt_translator.zero_grad()
    loss.backward()
    opt_translator.step()

print(f"English to Bengali Seq2Seq Model Trained with Word2Vec! Loss: {loss.item():.4f}")

# 3. Translation Inference Function
def translate_english_to_bengali(model, english_sentence):
    model.eval()
    tokens = [en_vocab["<SOS>"]] + [en_vocab.get(w, en_vocab["<UNK>"]) for w in english_sentence.split()] + [en_vocab["<EOS>"]]
    src_tensor = torch.tensor([tokens], dtype=torch.long).to(device)
    
    with torch.no_grad():
        hidden, cell = model.encoder(src_tensor)
        decoder_input = torch.tensor([[bn_vocab["<SOS>"]]], dtype=torch.long).to(device)
        
        translated_words = []
        for _ in range(max_bn_len):
            output, hidden, cell = model.decoder(decoder_input, hidden, cell)
            pred_idx = output.argmax(dim=-1).item()
            
            if pred_idx == bn_vocab["<EOS>"]:
                break
                
            translated_words.append(inv_bn_vocab.get(pred_idx, ""))
            decoder_input = torch.tensor([[pred_idx]], dtype=torch.long).to(device)
            
    return " ".join(translated_words)

print("\n--- Translation Test Outputs ---")
test_samples = ["i love my country", "i am learning nlp", "we are very happy", "i am very happy"]
for sample in test_samples:
    bengali_out = translate_english_to_bengali(translator, sample)
    print(f"English:  '{sample}'")
    print(f"Bengali:  '{bengali_out}'\n")

Encoder: Seq2SeqEncoder(
  (embedding): Embedding(34, 300, padding_idx=0)
  (lstm): LSTM(300, 64, batch_first=True)
)
Decoder: Seq2SeqDecoder(
  (embedding): Embedding(33, 32, padding_idx=0)
  (lstm): LSTM(32, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=33, bias=True)
)
Translator: Seq2SeqTranslation(
  (encoder): Seq2SeqEncoder(
    (embedding): Embedding(34, 300, padding_idx=0)
    (lstm): LSTM(300, 64, batch_first=True)
  )
  (decoder): Seq2SeqDecoder(
    (embedding): Embedding(33, 32, padding_idx=0)
    (lstm): LSTM(32, 64, batch_first=True)
    (fc): Linear(in_features=64, out_features=33, bias=True)
  )
)
English to Bengali Seq2Seq Model Trained with Word2Vec! Loss: 0.0011

--- Translation Test Outputs ---
English:  'i love my country'
Bengali:  'আমি আমার দেশকে ভালোবাসি'

English:  'i am learning nlp'
Bengali:  'আমি এনএলপি শিখছি'

English:  'we are very happy'
Bengali:  'আমরা খুব খুশি'

English:  'i am very happy'
Bengali:  'আমি এনএলপি শিখছি'

